## Exercise 1: Single-Layer Perceptron for AND Logic Gate

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
X = np.array([[0, 0],
              [0, 1],
              [1, 0],
              [1, 1]])

y = np.array([0, 0, 0, 1])

In [ ]:
# Step function (Activation function)
def step_function(z):
    if z > 0:
        return 1
    else:
        return 0

In [ ]:
# Perceptron Learning Algorithm
def train_perceptron(X, y, epochs=10, learning_rate=0.1):
    
    weights = [0.0, 0.0] # Initialize weights and bias to zero
    bias = 0.0

    for epoch in range(epochs):
        print(f"Epoch {epoch + 1}")
        for i in range(len(X)):
            
            z = weights[0] * X[i][0] + weights[1] * X[i][1] + bias   # Weighted sum: w1*x1 + w2*x2 + bias
            y_hat = step_function(z)                                 # Apply activation function to get prediction
            error = y[i] - y_hat                                     # Error = actual - predicted

            # Update each weight: wi = wi + lr * error * xi
            weights[0] = weights[0] + learning_rate * error * X[i][0]
            weights[1] = weights[1] + learning_rate * error * X[i][1]
            
            bias = bias + learning_rate * error                      # Update bias: b = b + lr * error
            print(f"  x={X[i]}, y={y[i]}, y_hat={y_hat}, error={error}, weights={weights}, bias={round(bias,2)}")

    return weights, bias

In [ ]:
# Predict function
def predict(X, weights, bias):
    predictions = []
    for i in range(len(X)):
        z = weights[0] * X[i][0] + weights[1] * X[i][1] + bias
        y_hat = step_function(z)
        predictions.append(y_hat)
    return predictions

In [ ]:
# Plot decision boundary
def plot_decision_boundary(X, y, weights, bias, title):
    x1_values = [-0.5, 1.5]
    x2_values = []

    for x1 in x1_values:
        if weights[1] != 0:
            x2 = -(weights[0] * x1 + bias) / weights[1]
        else:
            x2 = 0
        x2_values.append(x2)

    # Plot each data point
    for i in range(len(X)):
        if y[i] == 1:
            plt.scatter(X[i][0], X[i][1], color='blue', marker='o', s=100)
        else:
            plt.scatter(X[i][0], X[i][1], color='red', marker='x', s=100)

    # Plot decision boundary line
    plt.plot(x1_values, x2_values, color='green', label='Decision Boundary')
    plt.xlim(-0.5, 1.5)
    plt.ylim(-0.5, 1.5)
    plt.xlabel('x1')
    plt.ylabel('x2')
    plt.title(title)
    plt.legend()
    plt.grid(True)
    plt.show()

In [ ]:
# Run training and show results
weights, bias = train_perceptron(X, y, epochs=10, learning_rate=0.1)

print("\nFinal Weights:", weights)
print("Final Bias:", bias)

predictions = predict(X, weights, bias)
print("\nPredictions:", predictions)
print("Actual:     ", list(y))

plot_decision_boundary(X, y, weights, bias, title="AND Gate — Perceptron Decision Boundary")

## Exercise 2: Single-Layer Perceptron using PyTorch on Breast Cancer Dataset


In [ ]:
# pip install torch torchvision torchaudio  (if not already installed)
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

In [ ]:
from sklearn.datasets import load_breast_cancer

X, y = load_breast_cancer(return_X_y=True)

print("X shape:", X.shape)  # (569, 30) —> 30 input features
print("y shape:", y.shape)

In [ ]:
# Split into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
# Standardize the dataset
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [ ]:
# Convert data to PyTorch tensors
# reshape(-1, 1) converts target vector into a column vector
X_train = torch.tensor(X_train, dtype=torch.float32)
X_test = torch.tensor(X_test, dtype=torch.float32)

y_train = torch.tensor(y_train, dtype=torch.float32).reshape(-1, 1)
y_test = torch.tensor(y_test, dtype=torch.float32).reshape(-1, 1)

print(X_train.shape)
print(y_train.shape)
print(X_test.shape)
print(y_test.shape)

In [ ]:
# Build the neural network

class Perceptron(nn.Module):
    def __init__(self):
        super(Perceptron, self).__init__()   # calls parent class constructor
        self.linear = nn.Linear(30, 1)       # 30 input features, 1 output neuron

    def forward(self, x):
        x = self.linear(x)
        x = torch.sigmoid(x)                 # sigmoid activation for binary classification
        return x

model = Perceptron()

print(model)
for param in model.parameters():
    print(param)

In [ ]:
# Define Loss Function and Optimizer
criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters(), lr=0.01)

In [ ]:
# Train the model
epochs = 50

for epoch in range(epochs):
                                    
    predictions = model(X_train)            # Forward pass
    loss = criterion(predictions, y_train)  # Compute loss
    optimizer.zero_grad()                   # Reset gradients (PyTorch accumulates by default)
    loss.backward()                         # Back propagation —> compute gradients
    optimizer.step()                        # Update weights using Adam

    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch+1}, Loss: {loss.item():.4f}")

In [ ]:
# Model Evaluation
with torch.no_grad():  # do not track gradients inside this block
    y_pred_prob = model(X_test)
    y_pred = (y_pred_prob > 0.5).float()
    accuracy = (y_pred == y_test).sum().item() / y_test.size(0)

print("Accuracy:", accuracy)

In [ ]:
# Detailed metrics using scikit-learn
from sklearn.metrics import confusion_matrix, classification_report

y_pred_np = y_pred.numpy()
y_test_np = y_test.numpy()

print(confusion_matrix(y_test_np, y_pred_np))
print(classification_report(y_test_np, y_pred_np))